# Train the Sentinel-1 flood segmentation U-Net (Phase 1, model C)

Runs on a free Colab T4. Downloads Sen1Floods11 (cc-by-4.0) from the Hugging
Face mirror, trains the U-Net with `ml/sar/train_unet.py`, reports validation
IoU/Dice, and zips the artifacts for download into the repo's
`ml/artifacts/sar_unet/`.

**Setup:** put the repo's `ml/sar/` and `ml/requirements-ml.txt` on the Colab
runtime — either `git clone` (if you have a remote) or upload the `ml` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install torch segmentation-models-pytorch rasterio huggingface_hub

In [ ]:
%cd /content
# Either clone your remote or upload the ml folder. Example for a local upload:
import os
if not os.path.exists('/content/ml/sar/train_unet.py'):
    print('Upload the ml folder (File > Upload) then re-run this cell.')
    print('Or: !git clone <your-repo-url> && cd <repo>')

In [ ]:
# 1) Download + extract the dataset (~35 GB; needs the T4/TPU disk, not the
#    session RAM). Keep events representing a spread of basins incl. India.
!python ml/sar/download_sen1floods11.py --out /content/sen1floods11 \
    --keep-events TP02 Paraguay TP05 SriLanka TP07 Nepal TP08 India TP11 EastHarvest TP12 Caribbean

In [ ]:
# 2) Sanity: show tile / label pairs available.
tifs = !ls /content/sen1floods11/TP*/S1Hand_*.tif | head -20
tifs

In [ ]:
# 3) Train. First run is a quick fit to confirm the pipeline; bump --epochs
#    to 25-40 and --size to 256 for the real model (T4: ~40-80 s/epoch).
!python ml/sar/train_unet.py --mode sen1floods11 --data-dir /content/sen1floods11 \
    --epochs 3 --size 256 --batch-size 8 --encoder resnet18 --out /content/artifacts/sar_unet

In [ ]:
# 4) Report the measured numbers (this is what goes in the evidence sheet).
import json
meta = json.load(open('/content/artifacts/sar_unet/meta.json'))
print(f"val IoU = {meta['val_iou']:.4f}   val Dice = {meta['val_dice']:.4f}")
print(meta['note'])

In [ ]:
# 5) Download the artifacts, then drop them into ml/artifacts/sar_unet/ in
#    the repo (model.pt + meta.json). The backend picks them up automatically.
import shutil
shutil.make_archive('/content/sar_unet', 'zip', '/content/artifacts/sar_unet')
from google.colab import files
files.download('/content/sar_unet.zip')